# 11b — The necessity test (E19) on Alberta: leave-EFG-out anchors (T2) and the conditional no-EFG ensemble (T3)

Mirror of the parent's `18b_e19_solves` (study plan v0.17.2; AB spec v0.5 M16). Runs on the curated-block re-solve
(`VERSION = "v3.1"`, after 10 and 11). **T2** (12 × ~1 s at Alberta scale): for every design formulation, the anchor solved
with every EFG multiplier at 0 → `runs_v3.1/ab_l/A/e19_t2/<formulation_id>/run/portfolio.tif`. Core cells absent from every
no-EFG anchor are EFG-necessary by counterfactual (11c compares them with the adequacy-forced set, T1).
**T3** (~1 h at Alberta scale): the full no-EFG ensemble at the APPLIED band (anchor + 50 MGA members + 50 guarded
members per formulation) — runs ONLY if `spec/v3.1/e19_gate.json`, written by 11c, says the core is predominantly forced
(> 50%). Resumable; live internet (WLS). Kernel `R (y2y)`.

In [1]:
ANALYSIS <- "ab_y2y"
PROJ <- normalizePath(getwd())
while (!file.exists(file.path(PROJ, "config.py"))) {
  parent <- dirname(PROJ)
  if (identical(parent, PROJ)) stop("config.py not found above getwd() -- open from inside the repo")
  PROJ <- parent
}
setwd(PROJ)
source(file.path(PROJ, "prioritizr_core.R"))
source(file.path(PROJ, "mga_core.R"))
HERE <- file.path(PROJ, "analyses", "alberta_prioritization")
py <- file.path(PROJ, ".venv", "bin", "python")
code <- paste0("import config; print(config.write_manifest(analysis='", ANALYSIS, "', ",
               "handoff_dir=config.AB_HANDOFF_DIR, manifest_path=config.AB_HANDOFF_DIR/'manifest.json'))")
out <- suppressWarnings(system2(py, c("-c", shQuote(code)), stdout = TRUE, stderr = TRUE))
if (!is.null(attr(out, "status")) && attr(out, "status") != 0) stop(paste(out, collapse = "\n"))
mpath <- file.path(PROJ, "input_data", "aligned_stack_ab", "manifest.json")
VERSION <- "v3.1"      # the necessity test runs on the curated-block re-solve only (parent v0.17.2)
stopifnot("the necessity test (E19) runs on the curated-block re-solve only" = VERSION != "v1")
MANIFEST_REL <- sprintf("analyses/alberta_prioritization/spec/manifest_%s.csv", VERSION); FREEZE_REL <- sprintf("analyses/alberta_prioritization/spec/manifest_%s.sha256", VERSION)
REC_REL <- sprintf("analyses/alberta_prioritization/spec/%s", VERSION)
EFG_SUBDIR_EXPECTED <- paste0("iucn_efg_", sub("\\..*$", "", VERSION))
MAN <- read.csv(file.path(PROJ, MANIFEST_REL), stringsAsFactors = FALSE)
dig <- strsplit(readLines(file.path(PROJ, FREEZE_REL))[1], "  ")[[1]][1]
stopifnot(identical(unname(tools::sha256sum(file.path(PROJ, MANIFEST_REL))[[1]]), dig), nrow(MAN) == 12)
LEVEL <- unique(MAN$budget_level); stopifnot(length(LEVEL) == 1)
BUDGET_PCT <- unique(MAN$budget_pct); stopifnot(length(BUDGET_PCT) == 1)
RUNS_REL <- file.path(sprintf("analyses/alberta_prioritization/runs_%s/ab_l", VERSION), LEVEL); RUNS <- file.path(PROJ, RUNS_REL)
REAL245 <- "input_data/aligned_stack_ab/climate_realizations/macrorefugia_245_2071_2100.tif"
SC <- jsonlite::read_json(file.path(HERE, "spec", "scenarios_ab_v1.json")); BLOCKS <- lapply(SC$`_meta`$blocks, unlist)
FLOOR_G <- unique(MAN$floor_g); stopifnot(length(FLOOR_G) == 1)
ctx585 <- pr_setup(mpath, PROJ); ctx585 <- modifyList(ctx585, pr_ingest(ctx585))
ctx245 <- pr_setup(mpath, PROJ)
ctx245$layers$path[ctx245$layers$name == "climate_type_macrorefugia"] <- REAL245
ctx245 <- modifyList(ctx245, pr_ingest(ctx245))
efg_names <- ctx585$layers$name[ctx585$layers$role == "feature_efg"]
stopifnot(length(efg_names) > 0, all(grepl(paste0("/", EFG_SUBDIR_EXPECTED, "/"), ctx585$layers$path[ctx585$layers$role == "feature_efg"])))
base_for <- function(row, results_dir) {
  b <- if (grepl("^ssp245", row$climate_level)) ctx245 else ctx585
  b <- pr_override(b, budget_pct = BUDGET_PCT, results_dir = results_dir, results_subdir = "_base")
  modifyList(b, pr_planning_units(b))
}
form_wt  <- function(row) list(w = jsonlite::fromJSON(row$weight_vector), t = jsonlite::fromJSON(row$target_vector))
no_efg <- function(w) { for (f in efg_names) w[[f]] <- 0; w }        # every EFG multiplier -> 0 (the parent's E17 T3 convention)
# T2 solver settings: the parent relaxed its leave-EFG-out anchors to a 1e-3 witness gap with a 20-min cap because the EFG-free
# plateau took > 100 min at 1e-4 on 1.27 M cells (M4.28 addendum). At Alberta scale the anchors solve in seconds, so the
# PRE-REGISTERED 1e-4 gap is kept and only the cap is mirrored (disclosed, AB M18.2).
T2_GAP <- 1e-4; T2_TIME_LIMIT_S <- 1200
run_single <- function(row, w, t, out_rel, artifact = "run", opt_gap = T2_GAP, time_limit = T2_TIME_LIMIT_S) {
  done <- file.path(PROJ, out_rel, artifact, "run_summary.json")
  if (file.exists(done)) { cat(sprintf("   %s exists -- skipped\n", out_rel)); return(invisible(NULL)) }
  actx <- do.call(pr_override, c(list(base_for(row, out_rel), targets = t, feature_weight_multipliers = w,
      results_subdir = artifact, solver = "gurobi", decision_type = "binary", opt_gap = opt_gap, solver_time_limit = time_limit, portfolio_n = 1)))
  actx <- modifyList(actx, pr_weights(actx)); actx <- modifyList(actx, pr_targets(actx)); actx <- modifyList(actx, pr_penalty_matrices(actx))
  bp <- pr_build_problem(actx); actx$p <- bp$p; actx$solve_params <- bp$solve_params
  sv <- pr_solve(actx); actx$s <- sv$s; actx$timing <- sv$timing; actx$n_sol <- sv$n_sol; actx$sol_attrs <- sv$sol_attrs
  actx <- modifyList(actx, pr_summaries(actx)); pr_write_outputs(actx); invisible(NULL)
}
cat(sprintf("VERSION %s | level %s | %d design formulations | %d EFG features zeroed for the counterfactuals\n", VERSION, LEVEL, nrow(MAN), length(efg_names)))


prioritizr 8.1.0 | terra 1.9.34 | analysis=ab_y2y | solver=highs (single solution)
objective=min_shortfall | budget=30% target=100% | opt_gap=0.10 | time_limit=43200s
resolution: 1000 m (agg factor 1) | decisions=proportion
roi: mode=full | lock_in=pa_mask
penalties: connectivity=0 | boundary=0 | neighbor=0
outputs -> output_data/ab_y2y
ingested 21 features (8 continuous + 13 EFG) + cost + PA mask | grid 1286 x 3312 @ 1000 m
normalized 21 features to total=100000 each (scale-invariant conditioning)
prioritizr 8.1.0 | terra 1.9.34 | analysis=ab_y2y | solver=highs (single solution)
objective=min_shortfall | budget=30% target=100% | opt_gap=0.10 | time_limit=43200s
resolution: 1000 m (agg factor 1) | decisions=proportion
roi: mode=full | lock_in=pa_mask
penalties: connectivity=0 | boundary=0 | neighbor=0
outputs -> output_data/ab_y2y
ingested 21 features (8 continuous + 13 EFG) + cost + PA mask | grid 1286 x 3312 @ 1000 m
normalized 21 features to total=100000 each (scale-invariant condit

In [2]:
# ---- T2: leave-EFG-out anchors, one per design formulation (~1 s each at Alberta scale) ------------------------
for (i in seq_len(nrow(MAN))) {
  row <- MAN[i, ]; wt <- form_wt(row)
  cat(sprintf("== %s (%d/%d)\n", row$formulation_id, i, nrow(MAN)))
  run_single(row, no_efg(wt$w), wt$t, file.path(RUNS_REL, "e19_t2", row$formulation_id))
}
cat("T2 complete -- next: 11c_ab_e19_analysis (T1 + T2 agreement + the T3 gate)\n")


== s0_ssp585_theta5 (1/12)
  override budget_pct       -> 0.4470064
  override results_dir      -> analyses/alberta_prioritization/runs_v3.1/ab_l/A/e19_t2/s0_ssp585_theta5
  override results_subdir   -> _base
  EFFECTIVE targets: <none> | weight multipliers: <none>
  outputs  -> analyses/alberta_prioritization/runs_v3.1/ab_l/A/e19_t2/s0_ssp585_theta5/_base
planning units: 85,133 cells | budget = 45% = 38,055 cells
locked-in [pa_mask]: 27,972 cells (32.9% of window) -- fits within budget
  override targets          -> irrecoverable_carbon_m_soc=0.322, T6.1.web.map_v1.0=0.7193, T6.3.web.map_v1.0=0.4211, T6.2.web.alt_v2.0=0.3406, T4.4.web.orig_v1.0=0.3075, S1.1_SF1.1.web.merged_v3=0.2857, TF1.6_TF1.7.web.merged_v3=0.2091, SF1.2.web.orig_v1.0=0.2028, T5.1.web.mix_v1.0=0.1893, F1.3.web.map_v1.0=0.1837, T2.2.web.mix_v1.0=0.158, T6.4.web.orig_v1.0=0.1, F2.4.web.mix_v1.0=0.1, T2.1.web.mix_v1.0=0.1
  override feature_weight_multipliers -> climate_type_macrorefugia=1.038, transboundary_connectiv

A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (21 total)
│└•planning units:
│ ├•data:       <SpatRaster> (85133 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2206000, 273000, -920000, 3585000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 38055)
│├•penalties:   none specified
│├•features:
││├•targets:    relative targets (between 0.1 and 1)
││└•weights:    continuous values (between 0 and 2.281027)
│├•constraints: 
││└•1:          locked in constraints (27972 planning units)
│└•decisions:   binary decision
└•optimization
 ├•portfolio:   default portfolio
 └•solver:      gurobi solver (`gap` = 0.0001, `time_limit` = 1200, `first_feasible` = FALSE, …)
# ℹ Use `summary(...)` to see complete formulation.



Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2853457
Set parameter TimeLimit to value 1200
Set parameter NumericFocus to value 2
Set parameter Presolve to value 2
Set parameter Threads to value 10
WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 25.6.0 25G83)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Non-default parameters:
TimeLimit  1200
NumericFocus  2
Presolve  2
Threads  10

WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Optimize a model with 22 rows, 85154 columns and 1022923 nonzeros (Min)
Model fingerprint: 0x94accb71
Model has 8 linear objective coefficients
Variable types: 21 continuous, 85133 integer (85133 binary)
Coefficient statistics:
  Matrix range     [1e-03, 1e+05]
  Objective range  [2e-01, 2e+00]
  Bounds range     [1e+00, 1e+00]
  RHS r

A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (21 total)
│└•planning units:
│ ├•data:       <SpatRaster> (85133 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2206000, 273000, -920000, 3585000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 38055)
│├•penalties:   none specified
│├•features:
││├•targets:    relative targets (between 0.1 and 1)
││└•weights:    continuous values (between 0 and 2.401713)
│├•constraints: 
││└•1:          locked in constraints (27972 planning units)
│└•decisions:   binary decision
└•optimization
 ├•portfolio:   default portfolio
 └•solver:      gurobi solver (`gap` = 0.0001, `time_limit` = 1200, `first_feasible` = FALSE, …)
# ℹ Use `summary(...)` to see complete formulation.



Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2853457
Set parameter TimeLimit to value 1200
Set parameter NumericFocus to value 2
Set parameter Presolve to value 2
Set parameter Threads to value 10
WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 25.6.0 25G83)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Non-default parameters:
TimeLimit  1200
NumericFocus  2
Presolve  2
Threads  10

WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Optimize a model with 22 rows, 85154 columns and 1022923 nonzeros (Min)
Model fingerprint: 0x87877dec
Model has 8 linear objective coefficients
Variable types: 21 continuous, 85133 integer (85133 binary)
Coefficient statistics:
  Matrix range     [1e-03, 1e+05]
  Objective range  [1e-01, 2e+00]
  Bounds range     [1e+00, 1e+00]
  RHS r

...

Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2853457
Set parameter TimeLimit to value 1200
Set parameter NumericFocus to value 2
Set parameter Presolve to value 2
Set parameter Threads to value 10
WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 25.6.0 25G83)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Non-default parameters:
TimeLimit  1200
NumericFocus  2
Presolve  2
Threads  10

WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Optimize a model with 22 rows, 85154 columns and 1022923 nonzeros (Min)
Model fingerprint: 0x8019cdb9
Model has 8 linear objective coefficients
Variable types: 21 continuous, 85133 integer (85133 binary)
Coefficient statistics:
  Matrix range     [1e-03, 1e+05]
  Objective range  [8e-02, 3e+00]
  Bounds range     [1e+00, 1e+00]
  RHS r

A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (21 total)
│└•planning units:
│ ├•data:       <SpatRaster> (85133 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2206000, 273000, -920000, 3585000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 38055)
│├•penalties:   none specified
│├•features:
││├•targets:    relative targets (between 0.1 and 1)
││└•weights:    continuous values (between 0 and 2.079211)
│├•constraints: 
││└•1:          locked in constraints (27972 planning units)
│└•decisions:   binary decision
└•optimization
 ├•portfolio:   default portfolio
 └•solver:      gurobi solver (`gap` = 0.0001, `time_limit` = 1200, `first_feasible` = FALSE, …)
# ℹ Use `summary(...)` to see complete formulation.



Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2853457
Set parameter TimeLimit to value 1200
Set parameter NumericFocus to value 2
Set parameter Presolve to value 2
Set parameter Threads to value 10
WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 25.6.0 25G83)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Non-default parameters:
TimeLimit  1200
NumericFocus  2
Presolve  2
Threads  10

WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Optimize a model with 22 rows, 85154 columns and 1022923 nonzeros (Min)
Model fingerprint: 0xc1bf5942
Model has 8 linear objective coefficients
Variable types: 21 continuous, 85133 integer (85133 binary)
Coefficient statistics:
  Matrix range     [1e-03, 1e+05]
  Objective range  [3e-01, 2e+00]
  Bounds range     [1e+00, 1e+00]
  RHS r

A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (21 total)
│└•planning units:
│ ├•data:       <SpatRaster> (85133 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2206000, 273000, -920000, 3585000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 38055)
│├•penalties:   none specified
│├•features:
││├•targets:    relative targets (between 0.1 and 1)
││└•weights:    continuous values (between 0 and 10)
│├•constraints: 
││└•1:          locked in constraints (27972 planning units)
│└•decisions:   binary decision
└•optimization
 ├•portfolio:   default portfolio
 └•solver:      gurobi solver (`gap` = 0.0001, `time_limit` = 1200, `first_feasible` = FALSE, …)
# ℹ Use `summary(...)` to see complete formulation.



Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2853457
Set parameter TimeLimit to value 1200
Set parameter NumericFocus to value 2
Set parameter Presolve to value 2
Set parameter Threads to value 10
WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 25.6.0 25G83)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Non-default parameters:
TimeLimit  1200
NumericFocus  2
Presolve  2
Threads  10

WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Optimize a model with 22 rows, 85154 columns and 1022923 nonzeros (Min)
Model fingerprint: 0xab79d355
Model has 8 linear objective coefficients
Variable types: 21 continuous, 85133 integer (85133 binary)
Coefficient statistics:
  Matrix range     [1e-03, 1e+05]
  Objective range  [2e-01, 1e+01]
  Bounds range     [1e+00, 1e+00]
  RHS r

In [3]:
# ---- E17 T3 on the curated block: leave-one-theme-out anchors at S0 (5 solves, seconds at Alberta scale; parent M4.32) ----
# The parent re-solved its E17 leave-one-theme-out bars on v3.1 so nothing on the deck is a 40-class number. Mirrored here
# as the same five anchors (the four PROACT block-outs at 1e-4; the EFG-out = the T2 witness convention); 12 reads them for
# the per-theme latitude / composition shifts (no E17 one-pager at Alberta -- AB spec M11.4 stands; the shifts are a T-D line).
row0 <- MAN[MAN$formulation_id == "s0_ssp585_theta5", ]; wt0 <- form_wt(row0)
for (b in names(BLOCKS)) {
  w <- wt0$w; for (f in unlist(BLOCKS[[b]])) w[[f]] <- 0
  cat(sprintf("== %s OUT\n", b))
  run_single(row0, w, wt0$t, file.path(RUNS_REL, "e17_t3", paste0(b, "_out")), opt_gap = 1e-4, time_limit = 43200)
}
cat("== efg OUT\n")
run_single(row0, no_efg(wt0$w), wt0$t, file.path(RUNS_REL, "e17_t3", "efg_out"))
cat("E17 T3 (curated block, Alberta) complete -- 12 reads runs_v3.1/ab_l/A/e17_t3/\n")


== core_habitat OUT
  override budget_pct       -> 0.4470064
  override results_dir      -> analyses/alberta_prioritization/runs_v3.1/ab_l/A/e17_t3/core_habitat_out
  override results_subdir   -> _base
  EFFECTIVE targets: <none> | weight multipliers: <none>
  outputs  -> analyses/alberta_prioritization/runs_v3.1/ab_l/A/e17_t3/core_habitat_out/_base
planning units: 85,133 cells | budget = 45% = 38,055 cells
locked-in [pa_mask]: 27,972 cells (32.9% of window) -- fits within budget
  override targets          -> irrecoverable_carbon_m_soc=0.322, T6.1.web.map_v1.0=0.7193, T6.3.web.map_v1.0=0.4211, T6.2.web.alt_v2.0=0.3406, T4.4.web.orig_v1.0=0.3075, S1.1_SF1.1.web.merged_v3=0.2857, TF1.6_TF1.7.web.merged_v3=0.2091, SF1.2.web.orig_v1.0=0.2028, T5.1.web.mix_v1.0=0.1893, F1.3.web.map_v1.0=0.1837, T2.2.web.mix_v1.0=0.158, T6.4.web.orig_v1.0=0.1, F2.4.web.mix_v1.0=0.1, T2.1.web.mix_v1.0=0.1
  override feature_weight_multipliers -> climate_type_macrorefugia=0, transboundary_connectivity=0.30626

A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (21 total)
│└•planning units:
│ ├•data:       <SpatRaster> (85133 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2206000, 273000, -920000, 3585000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 38055)
│├•penalties:   none specified
│├•features:
││├•targets:    relative targets (between 0.1 and 1)
││└•weights:    continuous values (between 0 and 2.281027)
│├•constraints: 
││└•1:          locked in constraints (27972 planning units)
│└•decisions:   binary decision
└•optimization
 ├•portfolio:   default portfolio
 └•solver:      gurobi solver (`gap` = 0.0001, `time_limit` = 43200, `first_feasible` = FALSE, …)
# ℹ Use `summary(...)` to see complete formulation.



Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2853457
Set parameter TimeLimit to value 43200
Set parameter NumericFocus to value 2
Set parameter Presolve to value 2
Set parameter Threads to value 10
WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 25.6.0 25G83)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Non-default parameters:
TimeLimit  43200
NumericFocus  2
Presolve  2
Threads  10

WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Optimize a model with 22 rows, 85154 columns and 1022923 nonzeros (Min)
Model fingerprint: 0x2c3f2d80
Model has 20 linear objective coefficients
Variable types: 21 continuous, 85133 integer (85133 binary)
Coefficient statistics:
  Matrix range     [1e-03, 1e+05]
  Objective range  [8e-02, 2e+00]
  Bounds range     [1e+00, 1e+00]
  RH

A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (21 total)
│└•planning units:
│ ├•data:       <SpatRaster> (85133 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2206000, 273000, -920000, 3585000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 38055)
│├•penalties:   none specified
│├•features:
││├•targets:    relative targets (between 0.1 and 1)
││└•weights:    continuous values (between 0 and 2.281027)
│├•constraints: 
││└•1:          locked in constraints (27972 planning units)
│└•decisions:   binary decision
└•optimization
 ├•portfolio:   default portfolio
 └•solver:      gurobi solver (`gap` = 0.0001, `time_limit` = 43200, `first_feasible` = FALSE, …)
# ℹ Use `summary(...)` to see complete formulation.



Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2853457
Set parameter TimeLimit to value 43200
Set parameter NumericFocus to value 2
Set parameter Presolve to value 2
Set parameter Threads to value 10
WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 25.6.0 25G83)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Non-default parameters:
TimeLimit  43200
NumericFocus  2
Presolve  2
Threads  10

WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Optimize a model with 22 rows, 85154 columns and 1022923 nonzeros (Min)
Model fingerprint: 0xca8c372c
Model has 19 linear objective coefficients
Variable types: 21 continuous, 85133 integer (85133 binary)
Coefficient statistics:
  Matrix range     [1e-03, 1e+05]
  Objective range  [8e-02, 2e+00]
  Bounds range     [1e+00, 1e+00]
  RH

...

Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2853457
Set parameter TimeLimit to value 43200
Set parameter NumericFocus to value 2
Set parameter Presolve to value 2
Set parameter Threads to value 10
WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 25.6.0 25G83)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Non-default parameters:
TimeLimit  43200
NumericFocus  2
Presolve  2
Threads  10

WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Optimize a model with 22 rows, 85154 columns and 1022923 nonzeros (Min)
Model fingerprint: 0x3bedfd36
Model has 19 linear objective coefficients
Variable types: 21 continuous, 85133 integer (85133 binary)
Coefficient statistics:
  Matrix range     [1e-03, 1e+05]
  Objective range  [8e-02, 2e+00]
  Bounds range     [1e+00, 1e+00]
  RH

A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (21 total)
│└•planning units:
│ ├•data:       <SpatRaster> (85133 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2206000, 273000, -920000, 3585000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 38055)
│├•penalties:   none specified
│├•features:
││├•targets:    relative targets (between 0.1 and 1)
││└•weights:    continuous values (between 0 and 1.597804)
│├•constraints: 
││└•1:          locked in constraints (27972 planning units)
│└•decisions:   binary decision
└•optimization
 ├•portfolio:   default portfolio
 └•solver:      gurobi solver (`gap` = 0.0001, `time_limit` = 43200, `first_feasible` = FALSE, …)
# ℹ Use `summary(...)` to see complete formulation.



Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2853457
Set parameter TimeLimit to value 43200
Set parameter NumericFocus to value 2
Set parameter Presolve to value 2
Set parameter Threads to value 10
WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 25.6.0 25G83)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Non-default parameters:
TimeLimit  43200
NumericFocus  2
Presolve  2
Threads  10

WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Optimize a model with 22 rows, 85154 columns and 1022923 nonzeros (Min)
Model fingerprint: 0xa7a36ea5
Model has 19 linear objective coefficients
Variable types: 21 continuous, 85133 integer (85133 binary)
Coefficient statistics:
  Matrix range     [1e-03, 1e+05]
  Objective range  [8e-02, 2e+00]
  Bounds range     [1e+00, 1e+00]
  RH

A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (21 total)
│└•planning units:
│ ├•data:       <SpatRaster> (85133 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2206000, 273000, -920000, 3585000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 38055)
│├•penalties:   none specified
│├•features:
││├•targets:    relative targets (between 0.1 and 1)
││└•weights:    continuous values (between 0 and 2.281027)
│├•constraints: 
││└•1:          locked in constraints (27972 planning units)
│└•decisions:   binary decision
└•optimization
 ├•portfolio:   default portfolio
 └•solver:      gurobi solver (`gap` = 0.0001, `time_limit` = 1200, `first_feasible` = FALSE, …)
# ℹ Use `summary(...)` to see complete formulation.



Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2853457
Set parameter TimeLimit to value 1200
Set parameter NumericFocus to value 2
Set parameter Presolve to value 2
Set parameter Threads to value 10
WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 25.6.0 25G83)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Non-default parameters:
TimeLimit  1200
NumericFocus  2
Presolve  2
Threads  10

WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Optimize a model with 22 rows, 85154 columns and 1022923 nonzeros (Min)
Model fingerprint: 0x94accb71
Model has 8 linear objective coefficients
Variable types: 21 continuous, 85133 integer (85133 binary)
Coefficient statistics:
  Matrix range     [1e-03, 1e+05]
  Objective range  [2e-01, 2e+00]
  Bounds range     [1e+00, 1e+00]
  RHS r

In [4]:
# ---- T3 (CONDITIONAL): the no-EFG ensemble at the APPLIED band -- anchors + MGA + guarded members, EFG multipliers 0 -----
gate_f <- file.path(PROJ, REC_REL, "e19_gate.json")
gate <- if (file.exists(gate_f)) jsonlite::read_json(gate_f) else NULL
if (is.null(gate)) {
  cat("T3 gate not written yet -- run 11c_ab_e19_analysis first (it decides whether the core is predominantly forced)\n")
} else if (!isTRUE(gate$t3_triggered)) {
  cat(sprintf("T3 NOT triggered: forced share of the core %.1f%% (rule: > 50%%) -- the no-EFG ensemble is not run\n", 100 * gate$forced_share_core_all))
} else {
  G_APPLIED <- as.numeric(gate$applied_g); TAG <- sprintf("g%02d", round(100 * G_APPLIED))
  cat(sprintf("T3 TRIGGERED: forced share of the core %.1f%% -- solving the no-EFG ensemble at the applied band g = %.0f%% (~1 h at Alberta scale)\n", 100 * gate$forced_share_core_all, 100 * G_APPLIED))
  for (i in seq_len(nrow(MAN))) {
    row <- MAN[i, ]; wt <- form_wt(row); out_rel <- file.path(RUNS_REL, "e19_t3", row$formulation_id); cd <- file.path(PROJ, out_rel)
    dir.create(cd, recursive = TRUE, showWarnings = FALSE)
    cat(sprintf("\n===================== %s (%d/%d) =====================\n", row$formulation_id, i, nrow(MAN)))
    if (file.exists(file.path(cd, sprintf("mga_guard_%s.tif", TAG)))) { cat("   exists -- skipped\n"); next }
    actx <- pr_override(base_for(row, out_rel), targets = wt$t, feature_weight_multipliers = no_efg(wt$w), results_subdir = "mga_build",
                        solver = "gurobi", decision_type = "binary", opt_gap = 1e-4, portfolio_n = 1)
    actx <- modifyList(actx, pr_weights(actx)); actx <- modifyList(actx, pr_targets(actx)); actx <- modifyList(actx, pr_penalty_matrices(actx))
    bp <- pr_build_problem(actx); actx$p <- bp$p; actx$solve_params <- bp$solve_params
    cm <- mga_compile(actx); anchor <- mga_anchor(cm, opt_gap = row$opt_gap)
    if (!file.exists(file.path(cd, sprintf("mga_%s.tif", TAG)))) {
      gen <- mga_generate(cm, anchor, g = G_APPLIED, k = row$k_requested); mga_write(gen, cm, actx$cost, cd, TAG)
    }
    r <- terra::rast(actx$cost); v <- rep(NA_integer_, terra::ncell(r)); v[cm$pu_index] <- as.integer(anchor$x); terra::values(r) <- v
    terra::writeRaster(r, file.path(cd, "anchor.tif"), overwrite = TRUE, datatype = "INT1U", NAflag = 255, gdal = c("COMPRESS=DEFLATE", "TILED=YES"))
    jsonlite::write_json(list(formulation_id = row$formulation_id, experiment = "E19 T3 no-EFG ensemble (Alberta)", anchor_objective = anchor$z,
                              anchor_gap = anchor$gap, anchor_runtime_s = anchor$runtime, weight_vector = no_efg(wt$w), target_vector = wt$t,
                              k = row$k_requested, g = G_APPLIED, level = LEVEL, created_utc = format(Sys.time(), tz = "UTC")),
                         file.path(cd, "formulation_meta.json"), auto_unbox = TRUE, pretty = TRUE, digits = 10)
    gg <- mga_generate(cm, anchor, g = G_APPLIED, k = row$k_requested, floors = list(ctx = actx, blocks = BLOCKS, g = FLOOR_G))
    mga_write(gg, cm, actx$cost, cd, sprintf("guard_%s", TAG))
    cat(sprintf("   wrote anchor, MGA members, guarded members for %s\n", row$formulation_id))
  }
  cat("T3 complete -- re-run 11c_ab_e19_analysis for the F_noEFG surface\n")
}


T3 gate not written yet -- run 11c_ab_e19_analysis first (it decides whether the core is predominantly forced)
